# Data Visualization
1. **Prepare** data for visualization
2. **Analyze** number of rides by
   
   A. season

   B. month

   C. compare with weather conditions
   
3. **Visualize** Top Start Stations
4. **Visualize** Top End Stations

## Loading Libraries

In [1]:
import requests

import numpy as np
import pandas as pd

import plotly.express as px
import folium

import os
from pathlib import Path
from urllib.request import urlretrieve
from urllib.error import HTTPError, URLError
from zipfile import ZipFile

## Load Data

In [2]:
OUTPUT_DIR = "../data/JC/"

In [3]:
citibike_df = pd.read_csv(f"{OUTPUT_DIR}/JC2025_Enriched.csv", parse_dates=["started_at", "ended_at"])
citibike_df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,ride_duration_minutes,date,month,month_name,day_of_week,hour,season
0,9F734BE1BFC45FF4,electric_bike,2025-11-18 18:34:14.943,2025-11-18 18:47:33.391,Glenwood Ave,JC094,West Side Ave & Stegman Pkwy,JC131,40.727551,-74.071061,40.710870,-74.093680,member,13.307467,2025-11-18,2025-11,November,Tuesday,18,Autumn
1,B6C773B13AC0E465,classic_bike,2025-11-26 16:29:15.513,2025-11-26 16:43:45.235,Glenwood Ave,JC094,West Side Ave & Stegman Pkwy,JC131,40.727551,-74.071061,40.710870,-74.093680,member,14.495367,2025-11-26,2025-11,November,Wednesday,16,Autumn
2,C300465AA158280F,electric_bike,2025-11-04 22:31:58.010,2025-11-04 22:38:57.017,Bloomfield St & 15 St,HB203,Marshall St & 2 St,HB408,40.754530,-74.026580,40.740802,-74.042521,member,6.983450,2025-11-04,2025-11,November,Tuesday,22,Autumn
3,31A424FC97C8AAFB,classic_bike,2025-11-08 06:51:57.424,2025-11-08 06:57:45.627,Clinton St & 7 St,HB303,Marshall St & 2 St,HB408,40.745420,-74.033320,40.740802,-74.042521,member,5.803383,2025-11-08,2025-11,November,Saturday,6,Autumn
4,08C5EA04CB1FDC57,classic_bike,2025-11-24 20:31:21.758,2025-11-24 20:38:01.261,Clinton St & 7 St,HB303,Marshall St & 2 St,HB408,40.745420,-74.033320,40.740802,-74.042521,member,6.658383,2025-11-24,2025-11,November,Monday,20,Autumn


## Number of Rides

### Seasonal Data

In [4]:
season_rides = (
    citibike_df
    .groupby("season", as_index=False)
    .agg(
        number_of_rides=("ride_id", "count")
    )
)

season_order = ["Winter", "Spring", "Summer", "Autumn"]

season_rides["season"] = pd.Categorical(
    season_rides["season"],
    categories=season_order,
    ordered=True
)

season_rides = season_rides.sort_values("season")

season_rides

,season,number_of_rides
3,Winter,143472
1,Spring,247299
2,Summer,312111
0,Autumn,295399


In [5]:
fig = px.bar(
    season_rides,
    x="season",
    y="number_of_rides",
    title="Number of City Bike Rides per Season",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Season",
    yaxis_title="Number of Rides"
)

fig.show()

### Monthly Data

In [6]:
monthly_rides = (
    citibike_df
    .groupby("month", as_index=False)
    .agg(
        number_of_rides=("ride_id", "count")
    )
)

monthly_rides

,month,number_of_rides
0,2024-12,2
1,2025-01,50477
2,2025-02,45131
3,2025-03,73124
4,2025-04,81295
5,2025-05,92880
6,2025-06,96736
7,2025-07,107374
8,2025-08,108001
9,2025-09,115580


In [7]:
fig = px.bar(
    monthly_rides,
    x="month",
    y="number_of_rides",
    title="Number of Citi Bike Rides per Month"
)

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Number of Rides",
)

fig.show()

#### Highlight the Top Month

In [8]:
top_month = monthly_rides.loc[monthly_rides["number_of_rides"].idxmax(), "month"]

monthly_rides["highlight"] = np.where(
    monthly_rides["month"] == top_month,
    "Top Month",
    "Other Months"
)

monthly_rides

,month,number_of_rides,highlight
0,2024-12,2,Other Months
1,2025-01,50477,Other Months
2,2025-02,45131,Other Months
3,2025-03,73124,Other Months
4,2025-04,81295,Other Months
5,2025-05,92880,Other Months
6,2025-06,96736,Other Months
7,2025-07,107374,Other Months
8,2025-08,108001,Other Months
9,2025-09,115580,Top Month


In [9]:
fig = px.bar(
    monthly_rides,
    x="month",
    y="number_of_rides",
    color="highlight",
    text_auto=True,
    title="Highlighting Only the Top Month",
    color_discrete_map={
        "Top Month": "#1F77B4",
        "Other Months": "#B0BEC5"
    }
)

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Number of Rides",
    legend_title=""
)

fig.update_yaxes(showticklabels=False)

fig.show()

### Comparison with weather conditions

In [10]:
daily_rides = (
    citibike_df
    .groupby("date", as_index=False)
    .agg(
        number_of_rides=("ride_id", "count")
    )
)
daily_rides["date"] = pd.to_datetime(daily_rides["date"])
daily_rides.head()

,date,number_of_rides
0,2024-12-31,2
1,2025-01-01,1174
2,2025-01-02,1709
3,2025-01-03,1764
4,2025-01-04,1336


In [11]:
weather = pd.read_csv(f"{OUTPUT_DIR}/jersey_weather_2025.csv")
weather.info()

<class 'pandas.DataFrame'>
RangeIndex: 365 entries, 0 to 364
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   2m_max              365 non-null    float64
 1   2m_min              365 non-null    float64
 2   2m_mean             365 non-null    float64
 3   precipitation_sum   365 non-null    float64
 4   rain_sum            365 non-null    float64
 5   snowfall_sum        365 non-null    float64
 6   wind_speed_10m_max  365 non-null    float64
 7   date                365 non-null    str    
dtypes: float64(7), str(1)
memory usage: 22.9 KB


In [12]:
weather["date"] = pd.to_datetime(weather["date"])

In [13]:
bike_weather_daily = daily_rides.merge(
    weather,
    on="date",
    how="left"
)

bike_weather_daily.head()

,date,number_of_rides,2m_max,2m_min,2m_mean,precipitation_sum,rain_sum,snowfall_sum,wind_speed_10m_max
0,2024-12-31,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-01-01,1174,10.9,3.9,7.4,4.5,4.5,0.0,23.2
2,2025-01-02,1709,5.4,0.3,2.6,0.0,0.0,0.0,25.1
3,2025-01-03,1764,3.2,-1.9,0.4,0.0,0.0,0.0,17.1
4,2025-01-04,1336,-0.1,-2.7,-1.4,0.0,0.0,0.0,26.1


In [14]:
bike_weather_daily.isna().sum()

date                  0
number_of_rides       0
2m_max                1
2m_min                1
2m_mean               1
precipitation_sum     1
rain_sum              1
snowfall_sum          1
wind_speed_10m_max    1
dtype: int64

#### Rides and Average Temperature

In [15]:
fig = px.scatter(
    bike_weather_daily,
    x="2m_mean",
    y="number_of_rides",
    trendline="ols",
    title="Daily Rides vs Average Temperature"
)

fig.update_layout(
    xaxis_title="Average Daily Temperature",
    yaxis_title="Number of Rides"
)

fig.show()

#### Rides and Wind Speed

In [ ]:
fig = px.scatter(
    bike_weather_daily,
    x="wind_speed_10m_max",
    y="number_of_rides",
    trendline="ols",
    title="Daily Rides vs Maximum Wind Speed"
)

fig.update_layout(
    xaxis_title="Maximum Wind Speed",
    yaxis_title="Number of Rides"
)

fig.show()

#### Rides and Precipitation

In [17]:
fig = px.scatter(
    bike_weather_daily,
    x="precipitation_sum",
    y="number_of_rides",
    trendline="ols",
    title="Daily Rides vs Precipitation"
)

fig.update_layout(
    xaxis_title="Daily Precipitation",
    yaxis_title="Number of Rides"
)

fig.show()

#### Number of Rides vs Daily Average Temperature and Maximum Wind Speed 

In [48]:
fig = px.scatter_3d(
    bike_weather_daily, 
    x='2m_mean', 
    y='wind_speed_10m_max', 
    z='number_of_rides',
    color='number_of_rides', 
    opacity=0.7,
    labels={'number_of_rides': 'Number of Rides'},
    title='Daily Rides vs. Temperature and Wind Speed'
)

fig.update_layout(
    width=900,
    height=750, 
    margin=dict(l=50, r=50, t=50, b=50),
    
    scene=dict(
        xaxis=dict(title_text="Average Daily Temperature (°C)", title_font=dict(size=12)),
        yaxis=dict(title_text="Maximum Wind Speed", title_font=dict(size=12)),
        zaxis=dict(title_text="Daily Number of Rides", title_font=dict(size=12))
    )
)

fig.update_traces(
    hovertemplate="<br>".join([
        "<b>Temperature:</b> %{x}°C",
        "<b>Total Rides:</b> %{y}",
        "<b>Maximum Wind Speed:</b> %{z}" 
    ])
)


fig.show()

In [51]:
fig = px.scatter(
    bike_weather_daily, 
    x='2m_mean', 
    y='precipitation_sum',
    size='number_of_rides',
    color='number_of_rides', 
    color_continuous_scale=px.colors.sequential.Viridis,
    title='Daily Rides vs. Temperature and Precipitation'
)

fig.update_layout(
    xaxis_title="Daily Mean Temperature (°C)",
    yaxis_title="Precipitation (mm)",
    coloraxis_colorbar_title="Daily Number of Rides", 
)

fig.update_traces(
    hovertemplate="<br>".join([
        "<b>Temperature:</b> %{x}°C",
        "<b>Precipitation mm:</b> %{y}",
        "<b>Total Rides:</b> %{marker.color}" 
    ])
)


fig.show()

## Top Start Stations